# Tutorial 1: Environments and Neural Networks

**zrth** extracts symbolic reactive modules from Python classes. You write a standard [Gymnasium](https://gymnasium.farama.org/) environment or a [PyTorch](https://pytorch.org/) neural network, then wrap it with `Env()` or `Module()` — zrth analyzes it and produces a formal model with typed wires and terms that can be used for verification or training.

This tutorial walks through:
1. Defining an environment (plain `gym.Env`) and wrapping it with `zrth.gym.Env`
2. Defining a neural network (plain `nn.Module`) and wrapping it with `zrth.torch.Module`
3. Inspecting the extracted modules
4. Training a ranking function to prove a liveness property
5. Formally verifying the ranking conditions with Z3

## Defining an environment

Write a standard [Gymnasium](https://gymnasium.farama.org/) environment by subclassing `gym.Env`:

- **`__init__`**: set `action_space`, `observation_space`, any parameters (e.g. `self.y0`), and state variables (e.g. `self.x`)
- **`reset`**: initialize all state variables, return `(observation, info)`
- **`step`**: update state given an action, return `(observation, reward, terminated, truncated, info)`

**Important:** state variables must also be assigned in `__init__` so the analyzer can infer their types. For example, even though `reset` sets `self.x = 0.0`, you should also write `self.x = 0.0` in `__init__`.

Below is a counter system with three variables $x$, $y$, $z$. The variable $x$ increments while $x < y \lor x < z$, and resets to $0$ otherwise. We want to show that $x = y \lor x = z$ must occur infinitely often.

In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth.gym import Env
from zrth.torch import Module


class CounterEnv(gym.Env):
    """Counter environment: x increments while x < y or x < z, resets otherwise."""

    def __init__(self, y0, z0):
        super().__init__()

        self.action_space = spaces.Discrete(1)
        self.observation_space = spaces.Box(low=-1e6, high=1e6, shape=(1,))

        self.y0 = y0
        self.z0 = z0

        # Also set state variables here so the analyzer can infer their dtype
        self.x = 0.0
        self.y = y0
        self.z = z0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.x = 0.0
        self.y = self.y0
        self.z = self.z0

        observation = self.x
        return observation, {}

    def step(self, action):
        if self.x < self.y or self.x < self.z:
            self.x = self.x + 1.0
        else:
            self.x = 0.0

        # y and z remain constant
        self.y = self.y
        self.z = self.z

        at_target = self.x == self.y or self.x == self.z
        reward = 1.0 if at_target else 0.0
        terminated = at_target
        truncated = False
        observation = self.x
        return observation, reward, terminated, truncated, {}

## Wrapping and inspection

When you call `Env(counter_instance)`, zrth analyzes the class's `__init__`, `reset`, and `step` methods. It:
1. Infers action/observation types from the gymnasium spaces
2. Classifies `self.*` attributes as **private** (read+write state) or **parameters** (read-only constants)
3. Creates typed wire pairs for each variable and signal
4. Converts `reset` into **init** terms and `step` into **update** terms

Print the extracted module to see the result.

In [2]:
counter = CounterEnv(y0=30.0, z0=50.0)
wrapped_counter = Env(counter)
print(wrapped_counter)

module
  external
    w8, w9 : Real(1, 1)
  interface
    w10, w11 : Real(1, 1)
    w12, w13 : Real(1, 1)
    w14, w15 : Bool(1, 1)
    w16, w17 : Bool(1, 1)
  private
    w0, w1 : Real(1, 1)
    w2, w3 : Real(1, 1)
    w4, w5 : Real(1, 1)
  atom controls w1, w3, w5, w11, w13, w15, w17 reads w0, w2, w4
  init
    [[30.]]
Tensor[[1, 1], Float] w6; 
    [[50.]]
Tensor[[1, 1], Float] w7; 
    [[0.]]
Tensor[[1, 1], Float] w18; 
    Id w1; w18
    Id w3; w6
    Id w5; w7
    Id w11; w18
    [[0.]]
Tensor[[1, 1], Float] w13; 
    [[false]]
Tensor[[1, 1], Bool] w15; 
    [[false]]
Tensor[[1, 1], Bool] w17; 
  update
    [[30.]]
Tensor[[1, 1], Float] w6; 
    [[50.]]
Tensor[[1, 1], Float] w7; 
    Lt w19; w0, w2
    Lt w20; w0, w4
    [[false]]
Tensor[[1, 1], Bool] w21; 
    [[true]]
Tensor[[1, 1], Bool] w22; 
    Ite w23; w19, w22, w20
    [[1.]]
Tensor[[1, 1], Float] w24; 
    Add w25; w0, w24
    [[0.]]
Tensor[[1, 1], Float] w26; 
    Ite w27; w23, w25, w26
    Id w1; w27
    Id w3; w2
    

## Reading the output

The printed module shows:

- **external**: input wires — here, the action (unused by this environment, but still present)
- **interface**: output wires — observation, reward, terminated, truncated
- **private**: internal state — wire pairs `wN, wM` for x, y, z (each pair is `latched, next`)
- **init**: terms from `reset` — how each wire is initialized
- **update**: terms from `step` — how each wire is updated, including `Ite` (if-then-else), `Lt`, `Eq`, `Add`

## Defining a neural network

Write a standard `nn.Module`, then wrap it with `Module(instance)` to extract a symbolic module:

- **`__init__`**: define `nn.Linear` layers
- **`forward`**: standard PyTorch forward pass

zrth extracts the architecture as a **combinatorial** (stateless) module. The resulting module has:
- **extl** (external input) wire pair — sized from the first layer's `in_features`
- **intf** (interface output) wire pair — sized from the last layer's `out_features`

The ranking function below maps $(x, y, z) \in \mathbb{R}^3$ to a scalar. It is used in formal verification to prove that the liveness property $x = y \lor x = z$ holds infinitely often — not a controller, so we don't compose it with the counter.

In [3]:
class RankingNN(nn.Module):
    """Ranking function: R(x, y, z) -> scalar.

    Architecture: [3] -> 2 -> [1]
    """

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

Wrap and inspect. Note that the module has no `private` wires (it's stateless) and the `init`/`update` blocks are identical (combinatorial — same computation every step).

In [4]:
torch.manual_seed(2)

ranking = RankingNN()
wrapped_ranking = Module(ranking)
print(wrapped_ranking)

module
  external
    w37, w38 : Real(1, 3)
  interface
    w39, w40 : Real(1, 1)
  atom controls w40 awaits w38
  init
    Transpose w41; w38
    Linear w42; w41
    Transpose w43; w42
    ReLU w44; w43
    Transpose w45; w44
    Linear w46; w45
    Transpose w47; w46
    Id w40; w47
  update
    Transpose w41; w38
    Linear w42; w41
    Transpose w43; w42
    ReLU w44; w43
    Transpose w45; w44
    Linear w46; w45
    Transpose w47; w46
    Id w40; w47



## Training the ranking function

A **ranking function** $R: \mathcal{S} \to \mathbb{R}$ can be used to prove that a liveness property holds infinitely often. The key conditions are:

1. $R(s) \geq 0$ for all states $s$
2. $R(s') < R(s)$ whenever the property $x = y \lor x = z$ does *not* hold (the ranking strictly decreases)

If such an $R$ exists, the property must hold infinitely often — otherwise $R$ would decrease forever below zero, contradicting condition 1.

We train the `RankingNN` to approximate these conditions by:
1. Simulating the counter for $N$ steps (it's a fully functional `gym.Env`)
2. At each step, computing $R(x, y, z)$ and $R(x', y', z')$
3. Penalizing violations: $R$ should decrease at non-target states and be non-negative

In [5]:
optimizer = torch.optim.Adam(wrapped_ranking.parameters(), lr=0.01)
margin = 0.1
n_epochs = 201
n_steps = 40

# Fixed ordering of private wires for deterministic training
prvt_ordered = [wrapped_counter.get_prvt(n) for n in ('x', 'y', 'z')]

for epoch in range(n_epochs):
    wrapped_counter.reset()

    # Collect trajectory in fixed (x, y, z) order
    states = []
    for _ in range(n_steps):
        states.append(tuple(wrapped_counter._state[ltc].item() for ltc, _ in prvt_ordered))
        wrapped_counter.step(0)

    loss = torch.tensor(0.0)
    for i in range(len(states) - 1):
        s = torch.tensor(states[i], dtype=torch.float32)
        s_next = torch.tensor(states[i + 1], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze()
        r_next = wrapped_ranking(s_next.unsqueeze(0)).squeeze()

        # Problem-specific: define when the liveness property holds
        at_target = (wrapped_counter.x == wrapped_counter.y) or (wrapped_counter.x == wrapped_counter.z)

        if not at_target:
            loss = loss + torch.relu(r_next - r + margin)
            loss = loss + torch.relu(-r)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

epoch   0  loss = 285.8837
epoch  20  loss = 8.0702
epoch  40  loss = 7.1298
epoch  60  loss = 6.4479
epoch  80  loss = 5.6525
epoch 100  loss = 4.8636
epoch 120  loss = 4.0909
epoch 140  loss = 3.3021
epoch 160  loss = 2.4334
epoch 180  loss = 1.3859
epoch 200  loss = 0.0137


## Evaluating the trained ranking function

After training, we evaluate $R$ along a trajectory. At non-target states, $R$ should decrease on each step. These are approximate — formal verification follows below.

In [6]:
wrapped_counter.reset()

prvt_ordered = [wrapped_counter.get_prvt(n) for n in ('x', 'y', 'z')]
print(f"{'step':>4}  {'x':>5} {'y':>5} {'z':>5}  {'R(s)':>8}  {'target':>6}")
print("-" * 45)

with torch.no_grad():
    for step in range(102):
        vals = tuple(wrapped_counter._state[ltc].item() for ltc, _ in prvt_ordered)
        s = torch.tensor(vals, dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze().item()

        at_target = (wrapped_counter.x == wrapped_counter.y) or (wrapped_counter.x == wrapped_counter.z)

        print(f"{step:4d}  {vals[0]:5.1f} {vals[1]:5.1f} {vals[2]:5.1f}  {r:8.4f}  {'  *' if at_target else ''}")
        wrapped_counter.step(0)

step      x     y     z      R(s)  target
---------------------------------------------
   0    0.0  30.0  50.0    6.3995  
   1    1.0  30.0  50.0    6.2978  
   2    2.0  30.0  50.0    6.1961  
   3    3.0  30.0  50.0    6.0944  
   4    4.0  30.0  50.0    5.9926  
   5    5.0  30.0  50.0    5.8909  
   6    6.0  30.0  50.0    5.7892  
   7    7.0  30.0  50.0    5.6875  
   8    8.0  30.0  50.0    5.5858  
   9    9.0  30.0  50.0    5.4841  
  10   10.0  30.0  50.0    5.3824  
  11   11.0  30.0  50.0    5.2806  
  12   12.0  30.0  50.0    5.1789  
  13   13.0  30.0  50.0    5.0772  
  14   14.0  30.0  50.0    4.9755  
  15   15.0  30.0  50.0    4.8738  
  16   16.0  30.0  50.0    4.7721  
  17   17.0  30.0  50.0    4.6703  
  18   18.0  30.0  50.0    4.5686  
  19   19.0  30.0  50.0    4.4669  
  20   20.0  30.0  50.0    4.3652  
  21   21.0  30.0  50.0    4.2635  
  22   22.0  30.0  50.0    4.1618  
  23   23.0  30.0  50.0    4.0601  
  24   24.0  30.0  50.0    3.9583  
  25   25.0 

## Formal verification with Z3

Training gives us a *candidate* ranking function, but empirical checks on a single trajectory cannot guarantee that the conditions hold for *every* reachable state.

We can use an SMT solver ([Z3](https://github.com/Z3Prover/z3)) to verify the ranking conditions exhaustively. The idea: **negate** a condition and ask Z3 whether a counterexample exists. If Z3 returns `unsat`, no counterexample exists and the condition holds universally.

`zrth.smt.z3` translates module terms into Z3 expressions — the same term IR that `zrth.eval` executes with PyTorch tensors.

### Building the symbolic state

Create a Z3 variable for each latched private wire and execute the counter's update block symbolically. This gives us the next-state values as Z3 expressions.

In [7]:
import z3
from zrth import z3 as zz3

# Create Z3 variables in fixed (x, y, z) order
prvt_ordered = [wrapped_counter.get_prvt(n) for n in ('x', 'y', 'z')]
state = {latched: [z3.FreshInt()] for latched, _ in prvt_ordered}

# Execute the counter's update block term-by-term
for atom in wrapped_counter.atoms:
    for term in atom.update:
        read = [state[w] for w in term.read]
        state.update(zip(term.write, zz3.eval(term.itype, read)))

# Collect next-state values
next_state = {nxt: state[nxt] for _, nxt in prvt_ordered}

print("next-state expressions:")
for (_, nxt), name in zip(prvt_ordered, ('x', 'y', 'z')):
    print(f"  {name}' = {next_state[nxt]}")

next-state expressions:
  x' = [If(If(x!0 < x!1, True, x!0 < x!2), ToReal(x!0) + 1, 0)]
  y' = [x!1]
  z' = [x!2]


### Building the symbolic ranking function

Execute the ranking module's terms with Z3 to get $R(s)$ and $R(s')$ as Z3 expressions. The trained weights become exact rational constants in Z3.

In [8]:
extl = list(wrapped_ranking.extl)
intf = list(wrapped_ranking.intf)

def eval_ranking(input_vars):
    """Evaluate the ranking module's terms with Z3, returning R as a Z3 expression."""
    r_state = {extl[0][1]: input_vars}
    for atom in wrapped_ranking.atoms:
        for term in atom.update:
            read = [r_state[w] for w in term.read]
            r_state.update(zip(term.write, zz3.eval(term.itype, read)))
    return r_state[intf[0][1]][0]

# Build current/next state variable lists in fixed (x, y, z) order
current_vars = [v for ltc, _ in prvt_ordered for v in state[ltc]]
next_vars = [v for _, nxt in prvt_ordered for v in next_state[nxt]]

# R(s) and R(s')
R_s = eval_ranking(current_vars)
R_s_next = eval_ranking(next_vars)

# Problem-specific: grab Z3 variables for domain constraints
x = state[wrapped_counter.get_prvt('x')[0]][0]
y = state[wrapped_counter.get_prvt('y')[0]][0]
z = state[wrapped_counter.get_prvt('z')[0]][0]
domain = [y == 30, z == 50, z3.And(x >= 0, z3.Or(x < y, x < z))]

print("R(s) and R(s') built as Z3 expressions")

R(s) and R(s') built as Z3 expressions


### Verifying the conditions

We negate each condition and check satisfiability. If Z3 returns `unsat`, the condition holds for all reachable states. If `sat`, the model is a concrete counterexample.

**Condition 1:** $R(s) \geq 0$ for all states

In [9]:
# Condition 1: R(s) >= 0 for all reachable states
solver = z3.Solver()
solver.add(*domain)
solver.add(R_s < 0)          # negate: R(s) < 0

result = solver.check()
if result == z3.unsat:
    print("VERIFIED: R(s) >= 0 for all reachable states")
else:
    m = solver.model()
    print(f"COUNTEREXAMPLE: x={m.eval(x)}, y={m.eval(y)}, z={m.eval(z)}")
    print(f"  R(s) = {float(m.eval(R_s, model_completion=True).as_fraction()):.6f}")

VERIFIED: R(s) >= 0 for all reachable states


**Condition 2:** $R(s') < R(s)$ when not at target (strict decrease)

In [10]:
# Condition 2: R(s') < R(s) - margin/2 at all non-target states
solver = z3.Solver()
solver.add(*domain)
solver.add(R_s_next >= R_s - margin/2)     # negate: R does not decrease by margin/2

result = solver.check()
if result == z3.unsat:
    print("VERIFIED: R strictly decreases at all non-target states")
else:
    m = solver.model()
    print(f"COUNTEREXAMPLE: x={m.eval(x)}, y={m.eval(y)}, z={m.eval(z)}")
    r_val = float(m.eval(R_s, model_completion=True).as_fraction())
    r_next_val = float(m.eval(R_s_next, model_completion=True).as_fraction())
    print(f"  R(s) = {r_val:.6f}, R(s') = {r_next_val:.6f}")

VERIFIED: R strictly decreases at all non-target states


## Interactive module viewer

Finally, launch the interactive viewer to explore the counter module's wires and terms visually.

In [11]:
from zrth.visual import show
stop = show(wrapped_counter)

Module viewer running at http://127.0.0.1:52836 (ws://127.0.0.1:52837)
